# Gapfinder Prototype

This notebook prototypes the core flow for the Gapfinder application.

## Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
from IPython.display import display, Markdown

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"
ytt_api = YouTubeTranscriptApi()


## Step 1: Extract & Structure Knowledge
Fetch the transcript and extract key concepts.

In [2]:
def get_video_id(url):
    """Extract video ID from YouTube URL"""
    if "v=" in url:
        return url.split("v=")[1][:11]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1][:11]
    return url

def format_timestamp(seconds: float) -> str:
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

def get_transcript(video_id):
    """Download transcript"""
    transcript = ytt_api.fetch(video_id)
    return transcript.snippets


def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)


def process_yt_transcript(video_url):
    video_url = video_url
    video_id = get_video_id(video_url)
    transcript = get_transcript(video_id)
    subtitles = make_subtitles(transcript)
    return subtitles

def extract_concepts(transcript_text):
    """Use LLM to extract key concepts from the transcript"""
    prompt = f"""
    You are an expert educator. Extract the most important key concepts from the following transcript of an educational video.
    Return a bulleted list of 3-5 core concepts with a brief explanation for each.
    
    Transcript:
    {transcript_text}
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content


In [3]:
# Test Step 1
video_url = "https://www.youtube.com/watch?v=wjZofJX0v4M" # Example: 3Blue1Brown Transformers, the tech behind LLMs
video_id = get_video_id(video_url)
print(f"Video ID: {video_id}")

Video ID: wjZofJX0v4M


In [9]:
transcript = ytt_api.fetch(video_id)

In [14]:
transcript.snippets[0]

FetchedTranscriptSnippet(text='The initials GPT stand for Generative Pretrained Transformer.', start=0.0, duration=4.56)

In [4]:
transcript = get_transcript(video_id)

In [31]:
print(len(transcript))

transcript_text = ""

for s in transcript.snippets:
    # print(s.text)
    transcript_text += s.text + " "

print(transcript_text[:500])

443
The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. Pretrained refers to how the model went through a process of learning from a massive amount of data, and the prefix insinuates that there's more room to fine-tune it on specific tasks with additional training. But the last word, that's the real key piece. A transformer is a specific kind of neural network, a machine learning model, and it's the core i


In [ ]:
type)transcript_text_file)

In [29]:
transcript_text_file = Path(f'transcript_text_{video_id}.txt')
transcript_text = transcript_text_file.write_text(transcript_text)

In [52]:
subtitles = make_subtitles(transcript)

In [53]:
def process_yt_transcript(video_url):
    video_url = video_url
    video_id = get_video_id(video_url)
    transcript = get_transcript(video_id)
    subtitles = make_subtitles(transcript)
    return subtitles

In [56]:
subtitles = process_yt_transcript(video_url)

In [58]:
subtitles[:500]

"0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine "

In [25]:
subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles_file.write_text(subtitles)

31603

In [60]:
subtitles_file = Path(f'subtitles_{video_id}.txt')
subtitles = subtitles_file.read_text(encoding='utf-8')

In [61]:
print(subtitles[:500])

0:00 The initials GPT stand for Generative Pretrained Transformer.
0:05 So that first word is straightforward enough, these are bots that generate new text.
0:09 Pretrained refers to how the model went through a process of learning
0:13 from a massive amount of data, and the prefix insinuates that there's
0:16 more room to fine-tune it on specific tasks with additional training.
0:20 But the last word, that's the real key piece.
0:23 A transformer is a specific kind of neural network, a machine 


In [32]:
print("\nExtracting Concepts...")
concepts = extract_concepts(transcript_text)
display(Markdown(concepts))


Extracting Concepts...


- **Generative Pretrained Transformer (GPT)**: GPT stands for Generative Pretrained Transformer, which refers to a type of AI model that generates text. "Generative" indicates its ability to create new content, "pretrained" signifies that it has learned from a vast dataset before being fine-tuned for specific tasks, and "transformer" denotes the underlying neural network architecture that enables its capabilities.

- **Transformer Architecture**: The transformer is a specialized neural network model that processes data through a series of operations, including attention blocks and multi-layer perceptrons. It allows the model to understand context and relationships between words, making it effective for tasks like language translation and text generation.

- **Tokenization and Embedding**: Input data is broken down into smaller units called tokens, which can represent words or parts of words. Each token is then transformed into a vector through an embedding matrix, encoding its meaning in a high-dimensional space. This process allows the model to capture semantic relationships and contextual information.

- **Attention Mechanism**: The attention mechanism is a critical component of transformers that enables the model to weigh the importance of different tokens in a sequence. It allows the model to focus on relevant words when generating predictions, enhancing its understanding of context and improving the quality of generated text.

- **Probability Distribution and Softmax**: At the output stage, the model generates a probability distribution over possible next tokens using a softmax function. This function normalizes the raw output values (logits) into a valid probability distribution, allowing the model to predict the most likely next word based on the context provided.

## Chunking

In [65]:
from gitsource import chunk_documents

In [90]:
chunks = chunk_documents(
    [{'content': subtitles}],
    size=3000,
    step=500
)

In [91]:
print('num characters:', len(chunks[0]['content']))
print('num words:', len(chunks[0]['content'].split()))
print('num docs:', len(chunks))
print('chunk:', chunks[0])

num characters: 3000
num words: 538
num docs: 59
chunk: {'start': 0, 'content': "0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine learning model,\n0:27 and it's the core invention underlying the current boom in AI.\n0:31 What I want to do with this video and the following chapters is go through a\n0:35 visually-driven explanation for what actually happens inside a transformer.\n0:39 We're going to follow the data that flows through it and go step by step.\n0:43 There are many different kinds of models that you can build using transformer

In [92]:
from minsearch import Index

index = Index(text_fields=['content'])
index.fit(chunks)

In [94]:
index.docs[0]

{'start': 0,
 'content': "0:00 The initials GPT stand for Generative Pretrained Transformer.\n0:05 So that first word is straightforward enough, these are bots that generate new text.\n0:09 Pretrained refers to how the model went through a process of learning\n0:13 from a massive amount of data, and the prefix insinuates that there's\n0:16 more room to fine-tune it on specific tasks with additional training.\n0:20 But the last word, that's the real key piece.\n0:23 A transformer is a specific kind of neural network, a machine learning model,\n0:27 and it's the core invention underlying the current boom in AI.\n0:31 What I want to do with this video and the following chapters is go through a\n0:35 visually-driven explanation for what actually happens inside a transformer.\n0:39 We're going to follow the data that flows through it and go step by step.\n0:43 There are many different kinds of models that you can build using transformers.\n0:47 Some models take in audio and produce a transc

In [102]:
import random
from openai import OpenAI

client = OpenAI()

def generate_question(chunks, n_questions=10, max_chars=500):
    """
    Generiert Fragen basierend auf zufälligen Chunks.

    Args:
        chunks (list): Liste von Chunk-Dictionaries mit 'content'
        n_questions (int): Anzahl der gewünschten Fragen
        max_chars (int): Maximale Länge des Chunk-Textes im Prompt

    Returns:
        list: Liste von generierten Fragen
    """

    sampled_chunks = random.sample(chunks, min(n_questions, len(chunks)))

    questions = []

    for chunk in sampled_chunks:
        content = chunk.get("content", "")[:max_chars]

        prompt = f"""
        You will be given a text excerpt from a document.
        
        Formulate exactly ONE meaningful question that can be answered by this text.
        The question should:
        - be specific
        - clearly relate to the content
        - be well-suited for retrieval tests
        
        Text:
        <CONTEXT>
        {content}
        <CONTEXT>
        
        Frage:
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )

        question = response.choices[0].message.content.strip()
        questions.append(question)

    return questions

In [100]:
questions = generate_question(chunks)

In [101]:
questions

['What is the purpose of the constant T mentioned in the context of the distribution used by ChatGPT?',
 'What is the main goal of the process described in the text excerpt?',
 'What is the purpose of the system prompt in developing a chatbot based on GPT-3?',
 'What technique is being used to visualize word embeddings in the text?',
 'What is the primary goal in creating a model that predicts the next word according to the text?',
 'How can one find the word for a female monarch using the concept of embeddings?',
 'What is the number of parameters in the GPT-3 model mentioned in the text?',
 'What phenomenon occurs when a model adjusts its weights during training to create word embeddings with semantic meaning?',
 'What is the context size used for training GPT-3?',
 'What effect does having one significantly larger number in the input have on the normalized output distribution?']

In [103]:
question = questions[0]
answer = 'The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.'

In [104]:
print(question)
print(answer)

What is the purpose of the constant T mentioned in the context of the distribution used by ChatGPT?
The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.


In [106]:
import json
from openai import OpenAI
openai_client = OpenAI()

instructions = """
You are an expert tutor. Evaluate the user's answers to the QUESTION based on the the CONTEXT.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [ ]:
query = "python function definition"
usage, result = rag(query)

## Step 2: Generate Diagnostic Questions
Generate questions based on the concepts.

In [ ]:
def generate_questions(concepts):
    """Generate diagnostic questions based on concepts"""
    prompt = f"""
    Based on the following key concepts from a video, generate 3 diagnostic questions. 
    The questions should not be simple recall questions. They should be:
    1. A concept coverage question (testing understanding of the core mechanism).
    2. An "explain in your own words" prompt.
    3. An application question (transfer knowledge to a new scenario).
    
    Concepts:
    {concepts}
    
    Output the questions clearly numbered.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

print("Generating Questions...")
questions = generate_questions(concepts)
display(Markdown(questions))


## Step 3 & 4: User Answers & Gap Detection
Provide mock answers and detect gaps.

In [ ]:
# Step 3: Mock User Answers
mock_answers = """
1. A neural network is a machine learning model that takes some inputs, does some math, and gives an output. The weights are like importance scores.
2. The activation function is how the network decides if a neuron should fire. I think it uses a sigmoid function to squash values between 0 and 1.
3. If I change the image, the network will probably just output random numbers because it has not been trained on that specific image.
"""
display(Markdown(f"**User Answers:**\n{mock_answers}"))


In [108]:
question = questions[0]
answer = 'The constant , commonly referred to as Temperature in ChatGPT and other large language models (LLMs), is a hyperparameter used to control the randomness, "creativity," or "temperature" of the probability distribution used to select the next word (token) in a generated sequence.'

In [107]:
def detect_gaps_with_rag(question, answers):
    search_results = search(question)
    
    user_prompt = build_prompt(question, search_results)
    
    # erweitern um Antworten
    user_prompt += f"\n\n<USER_ANSWERS>\n{answers}\n</USER_ANSWERS>\n"
    
    user_prompt += """
    Evaluate the user's answers and provide:
    - What they understood well
    - What they misunderstood or missed
    - What to revisit
    """
    
    return llm(user_prompt, instructions)

In [110]:
gap_report = detect_gaps_with_rag(question, answer)

In [111]:
display(Markdown(gap_report))

### What the User Understood Well
- The user correctly identified the constant referred to as "Temperature" in the context of ChatGPT and other large language models (LLMs).
- They mentioned its role in controlling the randomness or creativity of the probability distribution used to select the next word (token), which captures the essence of how temperature affects sampling from the distribution.

### What They Misunderstood or Missed
- While the user mentioned that temperature controls "randomness," they could clarify the mechanism: a larger temperature results in a more uniform distribution, giving lower-probability words more weight, while a smaller temperature leads to higher-probability words dominating the selection process.
- They could also emphasize that in extreme cases (like T = 0), the model will always pick the highest probability token, which may lead to less diversity in the outputs.

### What to Revisit
- The user should revisit the specific effects of different temperature settings on the output distribution—how exactly higher or lower values influence the model's behavior.
- They might also want to explore the analogy of temperature with thermodynamic equations a bit more to understand why this term was chosen in the context of the model's probability distribution. 

Overall, the user's answer is strong but would benefit from additional detail and specificity regarding how temperature directly influences the behavior of the model's output.

In [ ]:
# Step 4: Gap Detection
def detect_gaps(concepts, questions, answers):
    prompt = f"""
    You are an expert tutor. Evaluate the user's answers to the diagnostic questions based on the core concepts of the video.
    
    Core Concepts:
    {concepts}
    
    Questions:
    {questions}
    
    User's Answers:
    {answers}
    
    Provide a structured report including:
    - **What they understood well:** Identify correct points in their answers.
    - **What they misunderstood or missed:** Clearly highlight the knowledge gaps or misconceptions.
    - **What to revisit:** Specifically state which concepts they need to review.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

print("Detecting Gaps...")
gap_report = detect_gaps(concepts, questions, mock_answers)
display(Markdown(gap_report))
